# NB_patient_360

Builds `adb_db_caresync.gold.patient_360` from the silver layer per the STTM (`Gold/sttm_silver_to_gold.xlsx`).

* **Grain:** one row per patient
* **Driving table:** `silver.patients` (no filters, all patients kept)
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id`, `silver.insurance_providers` on `insurance_provider_id`, `silver.lab_results` aggregated by `patient_id`
* **Load:** full rebuild with `CREATE OR REPLACE TABLE ... AS SELECT`
* **Note:** `patients.date_of_birth` is a timestamp in silver, so it is cast to `date` here

In [0]:
%sql
CREATE OR REPLACE TABLE adb_db_caresync.gold.patient_360 AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM adb_db_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                     AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                         AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name                                 AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)         AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT) AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT) AS critical_test_count,
  la.last_test_date
FROM adb_db_caresync.silver.patients p
LEFT JOIN adb_db_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_db_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id

In [0]:
%sql
-- Row count must equal silver.patients and patient_id must be unique
SELECT
  (SELECT COUNT(*) FROM adb_db_caresync.silver.patients) AS silver_patients,
  COUNT(*)                                               AS gold_rows,
  COUNT(DISTINCT patient_id)                             AS distinct_patients,
  SUM(total_tests)                                       AS total_tests,
  (SELECT COUNT(*) FROM adb_db_caresync.silver.lab_results) AS silver_lab_results
FROM adb_db_caresync.gold.patient_360

In [0]:
updated_df = spark.table("adb_db_caresync.gold.patient_360")
record_count = updated_df.count()

dbutils.notebook.exit(str(record_count))